# ZipEats pitch claims: mean intervals from 50 raw orders

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Confidence Interval Estimation for the Mean** · a worked example from the StatNex teaching guide

**The story.** ZipEats wants to put "typical order: about \$15, delivered in about 25 minutes" in a pitch to new restaurants. Its 50 sampled orders are only a sample of thousands, so the claims need a margin of error.

**The question.** What range for ZipEats' true average delivery time, order total and tip fits the data?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_mean_ci.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Delivery time (min),Distance (km),Order total ($),Tip ($)
1.0,16.0,1.5,18.41,2.07
2.0,24.0,1.8,22.84,2.68
3.0,28.0,2.4,12.55,1.77
4.0,19.0,1.0,14.25,0.0
5.0,29.0,1.9,14.64,0.21
6.0,26.0,1.9,16.77,1.34
7.0,22.0,1.2,17.94,2.51
8.0,30.0,2.9,9.43,0.0
9.0,22.0,1.3,12.85,2.08
10.0,26.0,2.1,19.0,1.79
11.0,28.0,1.2,21.85,2.97
12.0,17.0,1.6,9.97,0.0
13.0,29.0,2.5,21.83,3.46
14.0,26.0,2.3,14.55,0.89
15.0,23.0,1.3,17.27,1.06
16.0,23.0,1.5,21.22,1.76
17.0,23.0,2.2,15.01,0.95
18.0,22.0,1.8,17.59,2.45
19.0,26.0,1.4,14.39,0.86
20.0,12.0,0.7,18.94,0.64
21.0,26.0,2.5,17.0,2.43
22.0,14.0,2.0,9.05,1.5
23.0,29.0,3.2,13.19,0.0
24.0,20.0,2.3,11.02,0.99
25.0,23.0,2.1,20.25,0.63
26.0,20.0,1.3,16.26,1.03
27.0,27.0,3.2,18.54,1.01
28.0,23.0,2.2,14.43,2.45
29.0,28.0,1.8,13.04,0.67
30.0,19.0,2.7,9.3,0.76
31.0,27.0,2.3,14.7,0.02
32.0,27.0,2.5,11.43,0.98
33.0,29.0,3.1,11.19,0.55
34.0,24.0,1.8,19.65,0.81
35.0,26.0,1.6,18.99,2.62
36.0,18.0,1.1,20.07,0.83
37.0,25.0,2.2,12.95,2.23
38.0,58.0,2.0,15.34,1.43
39.0,17.0,1.0,10.77,0.0
40.0,31.0,2.6,20.83,0.22
41.0,27.0,2.2,13.66,1.75
42.0,18.0,1.3,12.59,0.42
43.0,24.0,1.4,11.63,0.55
44.0,29.0,1.6,12.33,1.37
45.0,25.0,3.1,12.61,0.74
46.0,24.0,1.2,14.06,1.97
47.0,25.0,2.3,19.86,1.34
48.0,23.0,2.6,12.98,2.01
49.0,20.0,0.7,14.33,0.7
50.0,26.0,1.6,9.85,1.46

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
confidence = 95 / 100

## Confidence interval for the mean, one column at a time

In [ ]:
# The interval is mean ± multiplier × standard error. The standard error, SD / √n, is
# how far the average of a sample this size typically lies from the true average. Read
# the interval as the range of true averages these data are consistent with: it says
# nothing about where single values fall, which the SD describes.
results = {}
for name in variables:
    # Text and blank cells count as missing and are skipped.
    x = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
    n, mean, sd = len(x), x.mean(), x.std(ddof=1)
    se = sd / np.sqrt(n)
    # The t distribution with n - 1 degrees of freedom, because the SD is estimated from
    # the same data: its multiplier is a little above the normal 1.96 at 95%. It assumes
    # independent values from a roughly normal population, or enough of them for the
    # average to be close to normal (the central limit theorem).
    lower, upper = stats.t.interval(confidence, n - 1, loc=mean, scale=se)
    results[name] = {"n": n, "variance": sd**2, "SD": sd, "SE": se,
                     "lower": lower, "mean": mean, "upper": upper}

table = pd.DataFrame(results)
print(table)
print(f"\nConfidence level: {confidence:.1%}")

## What the result says

The 50 sampled orders took 24.46 minutes to deliver on average. A sample mean is only an estimate, so the result adds a confidence interval: a range of values the data agree with for the true average of all orders. At the 95% confidence level it runs from 22.6 to 26.3 minutes. The standard error, 0.92 minutes, is how far a sample mean typically lies from the true mean.

This interval includes 25 minutes, so the delivery claim fits the data. The mean order total is \$15.26, with an interval from \$14.19 to \$16.34, which includes \$15.

Both pitch claims hold up. But these ranges describe the average order, not any single order.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.